# MiniVaultDB Analytics — EDA & Model Training

This notebook demonstrates the full analytics pipeline:
1. Ingest a dataset into MiniVaultDB
2. Retrieve and inspect it
3. Exploratory Data Analysis
4. Preprocessing
5. Model Training & Evaluation

In [ ]:
import sys, warnings
sys.path.insert(0, '..')
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, roc_auc_score, classification_report,
    ConfusionMatrixDisplay, RocCurveDisplay
)
import joblib

from pipeline.ingest import ingest_csv, download_sample_dataset
from pipeline.retrieve import retrieve_to_dataframe
from pipeline.preprocess import preprocess

plt.rcParams['figure.dpi'] = 110
sns.set_theme(style='whitegrid', palette='muted')
print('Setup complete.')

## 1 — Ingest dataset into MiniVaultDB

In [ ]:
# Downloads Titanic dataset and ingests it
# Change csv_path to your own dataset if needed
csv_path = download_sample_dataset('../data/titanic.csv')
n = ingest_csv(csv_path, db_path='../vault_data', key_prefix='passenger', id_column='PassengerId')
print(f'Ingested {n} records.')

## 2 — Retrieve from MiniVaultDB

In [ ]:
df = retrieve_to_dataframe(db_path='../vault_data')
print(f'Shape: {df.shape}')
df.head()

## 3 — Exploratory Data Analysis

In [ ]:
print('=== Basic info ===')
print(df.dtypes)
print('\n=== Missing values ===')
missing = df.isnull().sum()
print(missing[missing > 0])
print('\n=== Descriptive stats ===')
df.describe()

In [ ]:
# Target distribution
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

df['Survived'].value_counts().plot.bar(ax=axes[0], color=['#E24B4A','#1D9E75'])
axes[0].set_title('Survival count')
axes[0].set_xticklabels(['Did not survive', 'Survived'], rotation=0)

df['Survived'].value_counts(normalize=True).plot.pie(
    ax=axes[1], autopct='%1.1f%%',
    colors=['#E24B4A','#1D9E75'], labels=['Did not survive','Survived']
)
axes[1].set_ylabel('')
axes[1].set_title('Survival rate')
plt.tight_layout()
plt.show()

In [ ]:
# Numeric distributions
num_cols = df.select_dtypes(include='number').columns.tolist()
num_cols = [c for c in num_cols if c not in ['PassengerId','Survived','__key']]

n_cols = 3
n_rows = (len(num_cols) + n_cols - 1) // n_cols
fig, axes = plt.subplots(n_rows, n_cols, figsize=(14, 4 * n_rows))
axes = axes.flatten()

for i, col in enumerate(num_cols):
    df[col].dropna().hist(ax=axes[i], bins=30, color='#378ADD', edgecolor='white')
    axes[i].set_title(col)

for j in range(i+1, len(axes)):
    axes[j].set_visible(False)

plt.suptitle('Numeric feature distributions', y=1.01, fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# Survival by sex and class
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

df.groupby('Sex')['Survived'].mean().plot.bar(ax=axes[0], color=['#534AB7','#1D9E75'])
axes[0].set_title('Survival rate by sex')
axes[0].set_ylabel('Rate')
axes[0].set_xticklabels(axes[0].get_xticklabels(), rotation=0)

df.groupby('Pclass')['Survived'].mean().plot.bar(ax=axes[1], color=['#EF9F27','#BA7517','#854F0B'])
axes[1].set_title('Survival rate by passenger class')
axes[1].set_xticklabels(['1st','2nd','3rd'], rotation=0)

plt.tight_layout()
plt.show()

In [ ]:
# Age distribution by survival
fig, ax = plt.subplots(figsize=(10, 4))
df[df['Survived']==0]['Age'].dropna().plot.kde(ax=ax, label='Did not survive', color='#E24B4A')
df[df['Survived']==1]['Age'].dropna().plot.kde(ax=ax, label='Survived', color='#1D9E75')
ax.set_title('Age distribution by survival')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Correlation heatmap (numeric cols only)
corr_df = df[num_cols + ['Survived']].dropna()
corr = corr_df.corr()

fig, ax = plt.subplots(figsize=(8, 6))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(
    corr, mask=mask, annot=True, fmt='.2f',
    cmap='RdYlGn', center=0, ax=ax,
    linewidths=0.5, square=True
)
ax.set_title('Feature correlation matrix')
plt.tight_layout()
plt.show()

## 4 — Preprocessing

In [ ]:
TARGET = 'Survived'
DROP   = ['Name', 'Ticket', 'Cabin', 'PassengerId']

X_train, X_test, y_train, y_test, preprocessor = preprocess(
    df,
    target_col=TARGET,
    drop_cols=DROP,
    test_size=0.2,
    random_state=42,
)
print(f'Train: {X_train.shape}  |  Test: {X_test.shape}')

## 5 — Train & Evaluate

In [ ]:
model = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1)
model.fit(X_train, y_train)

y_pred  = model.predict(X_test)
y_proba = model.predict_proba(X_test)[:, 1]

acc = accuracy_score(y_test, y_pred)
auc = roc_auc_score(y_test, y_proba)

print(f'Accuracy : {acc:.4f}')
print(f'ROC-AUC  : {auc:.4f}')
print(f'\n{classification_report(y_test, y_pred, target_names=["Did not survive","Survived"])}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

ConfusionMatrixDisplay.from_predictions(
    y_test, y_pred,
    display_labels=['Did not survive','Survived'],
    cmap='Blues', ax=axes[0]
)
axes[0].set_title('Confusion matrix')

RocCurveDisplay.from_predictions(y_test, y_proba, ax=axes[1], color='#534AB7')
axes[1].set_title(f'ROC curve (AUC = {auc:.3f})')
axes[1].plot([0,1],[0,1],'--', color='gray')

plt.tight_layout()
plt.show()

In [ ]:
# Feature importances
try:
    num_names  = preprocessor.transformers_[0][2]
    ohe        = preprocessor.named_transformers_.get('cat')
    cat_names  = list(ohe.named_steps['encoder'].get_feature_names_out()) if ohe else []
    feat_names = list(num_names) + cat_names
    importances = model.feature_importances_
    feat_df = pd.DataFrame({'feature': feat_names, 'importance': importances})
    feat_df = feat_df.sort_values('importance', ascending=False).head(15)

    fig, ax = plt.subplots(figsize=(10, 5))
    ax.barh(feat_df['feature'][::-1], feat_df['importance'][::-1], color='#534AB7')
    ax.set_title('Top 15 feature importances')
    ax.set_xlabel('Importance')
    plt.tight_layout()
    plt.show()
except Exception as e:
    print(f'Could not plot feature importances: {e}')

In [ ]:
import pathlib
pathlib.Path('../models').mkdir(exist_ok=True)
joblib.dump(model, '../models/model_rf.pkl')
joblib.dump(preprocessor, '../models/preprocessor.pkl')
print('Model and preprocessor saved to models/')